# Un esempio di catalisi con MACE: Elettro-ossidazione di CO su Cu

Realizzato dal Laboratorio di Modellazione Multiscala del Politecnico di Torino (Italia). Queste risorse sono destinate a scopi didattici e sono state progettate per il corso di laurea triennale: "Applicazioni Energetiche dei materiali", tenuto al Politecnico di Torino nel secondo semestre 2026.

Michele Pellegrino (michele.pellegrino@polito.it)

I dati utilizzati come input sono tratti dalla [repository Zenodo](https://zenodo.org/records/17155822) associata alla pubblicazione:

[Ilyes Batatia et al., **A foundation model for atomistic materials chemistry**, J. Chem. Phys. 163, 184110 (2025)](https://doi.org/10.1063/5.0297006)

## Motivazione applicativa

<img src="https://upload.wikimedia.org/wikipedia/commons/2/22/Fuel_cell_NASA_p48600ac.jpg" width="175"/>

_Direct methanol fuel cell_ (fonte: [Wikipedia](https://en.wikipedia.org/wiki/Direct_methanol_fuel_cell))

### Pile a combustibile

Le PEMFC (_Proton Exchange Membrane Fuel Cells_) sono tecnologicamente interessanti per via delle loro proprietà di compattezza e portabilità, dovute alla bassa temperatura di esercizio, alla leggerezza dei materiali, all’elevata densità energetica e alla longevità operativa.

Settori di interesse:
- Trasporti (idrogeno, biocombustibili);
- Dispositivi portatili (e.g. _off-grid_);
- UPS (_Uninterrupted Power Supply_).

### Cu vs. Au, Pt

Il rame rappresenta un'alternativa economica ad altri metalli catalitici, come oro e platino. Inoltre, rispetto al platino, può risultare meno suscettibile al _CO poisoning_, ossia alla diminuzione dell’attività catalitica dovuta all’adsorbimento di CO.

## Motivazione scientifica

<img src="https://pubs.acs.org/cms/10.1021/acsenergylett.0c01751/asset/images/medium/nz0c01751_0004.gif" width="275"/>

Fonte: https://pubs.acs.org/doi/10.1021/acsenergylett.0c01751

### Interfacce metalliche

Esistono buoni modelli classici per molecole organiche (TIP4P, OPLS-AA, ...) ed esistono buoni modelli semi-empirici per i metalli in fase solida "bulk" (e.g. Embedded Atom Models). Tuttavia, non esistono potenziali di meccanica molecolare capaci di **modellare simultaneamente** molecole organiche e metalli all’interfaccia.

L’adsorbimento di acqua e altre piccole molecole su superfici metalliche non è banale! Non può essere modellato tramite un semplice potenziale di coppia, in stile Lennard-Jones.

### Reazioni chimiche

Le reazioni chimiche comportano la rottura e la formazione di legami, quindi non sono modellabili tramite potenziali di meccanica molecolare classici. Servono **potenziali reattivi** (ReaxFF, COMB, ...), parametrizzati _ad hoc_ e tipicamente poco accurati.

### Neural Network Potentials

La capacità di incorporare spazi chimici differenti e determinare automaticamente la topologia molecolare (i.e. legami, angoli, ...) rende i NNPs ideali per questo tipo di applicazioni.

## Installazione di PyTorch e MACE

<img src="https://upload.wikimedia.org/wikipedia/commons/9/96/Pytorch_logo.png?_=20211003060202" width="150"/>

Se non avete una GPU NVIDIA, installate PyTorch per CPU:

In [ ]:
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cpu

**Se il vostro laptop possiede una GPU NVIDIA**, potete sfruttare la sua potenza di calcolo installando PyTorch con supporto CUDA. Controllate la disponibilità della GPU e di CUDA eseguendo:

In [ ]:
!nvcc --version

In [ ]:
!nvidia-smi

Se disponete di CUDA$\ge$12.6, installate seguendo le istruzioni che trovate qui: https://pytorch.org/get-started/locally/

In [ ]:
# In sostanzam sostituite la versione di CUDA a <XYZ> la versione
# CUDA 12.6 -> cu126
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu<XYZ>

Se invece avete una versione di CUDA più vecchia, potete comunque cercare una versione compatibile di PyTorch qui: https://pytorch.org/get-started/previous-versions/ (sconsigliato). Oppure usate la CPU (consigliato).

<img src="https://avatars.githubusercontent.com/u/68508620?s=200&v=4" width="75"/>

Per ottenere la versione di MACE più recente, eseguite:

In [ ]:
!pip install mace-torch cuequivariance

La libreria `cuequivariance` non è strettamente necessaria; serve sostenzialmente ad accelerare la valutazione della rete neurale sfuttando la sua struttura equivariante. Funziona solo in combinazione con CUDA, quindi non viene usata nel caso PyTorch sia installato per CPU.

Controllate che le librerie siano state installate ispezionando l'output di `pip list`

In [ ]:
!pip list

## ASE

<img src="https://ase-lib.org/_static/ase256.png" width="75"/>

In questo laboratorio useremo [**A**tomistic **S**imulation **E**nvironment (**ASE**)](https://ase-lib.org/). ASE è un "orchestratore", ovvero un software che si appoggi a "kernels" esterni per eseguire simulazioni molecolari. L'idea è che:
- I kernels (i.e. **calcolatori** di interazioni tra interatomiche) sono tipicamente difficili da implementare, ma fanno l'_heavy lifting_ nelle simulazioni molecolari (i.e. prendono la maggior parte del tempo).
- Se "affido" il calcolo delle interazioni ad un kernel esterno, posso concentrarmi su tasks meno costose da un punto di vista computazionale (pre- e post-processing, updates di coodinate e velocità, visualizzazione, ...).

**Pro**:
- ASE può sfruttare kernels super-ottimizzati (e.g. GROMACS) o molto specifici (e.g. MACE) nello stesso ambiente, ottimo per testare approcci differenti;
- Completamente utilizzabile tramite API Python.

**Contro**:
- Non è tanto efficiente quanto un software completamente integrato e non scala su supercomputers (HPC);
- Implementa pochi metodi di post-processing, spesso bisogna analizzare le simulazioni "a mano" o usando altri moduli (e.g. [MDAnalysis](https://www.mdanalysis.org/)).

## Lettura e visualizzazione di una traiettoria

In [ ]:
# Moduli Python (ci serviranno più avanti...)
import matplotlib.pyplot as plt
import numpy as np
import copy
import time
import sys

Come prima cosa, per prendere confidenza con ASE, leggeremo e visualizzeremo una traiettoria già simulata. La traiettoria mostra il primo step della reazione di elettro-ossidazione:

$$ \mbox{OH}^*+\mbox{CO}^* \rightleftharpoons \mbox{COOH}^* $$

che vedremo percorrere da $\mbox{CO}^*+\mbox{OH}^*$ a $\mbox{COOH}^*$. L'apice $*$ indica che il composto è adsorbito sulla superficie di $\mbox{Cu}$.

Innanzitutto ispezioniamo il file contenente le traiettoria e leggiamolo con ASE:

In [ ]:
from ase import Atoms
from ase.io import read

trajectory_file_CO_OH = "COoxCu_111_CO-OH.extxyz"

### TODO: Per la relazione dovrete caricare una traiettoria differente ###
# trajectory_file_CO_OH = ...

# Usiamo il metodo read() per leggere la traiettoria da file di testo
# index=':' e' per indicare che vogliamo leggere tutta la traiettoria, non un solo frame
trajectory_catalysis = read(trajectory_file_CO_OH,format='extxyz',index=':')

Le informazioni sul formato della traiettoria sono consultabili qui: https://www.ovito.org/manual/reference/file_formats/input/xyz.html#file-formats-input-xyz-extended-format

In [ ]:
!gedit {trajectory_file_CO_OH}

Visualizziamo la traiettoria con lo strumento built-in di ASE usando il metodo `view()`:

In [ ]:
from ase.visualize import view

view(trajectory_catalysis)

Selezioniamo ora un singolo frame, per ispezionarlo nel dettaglio:

In [ ]:
# index=-1 usa la stessa sintassi delle liste
# I.e. siamo selezionando l'ultimo frame
atoms_catalysis = read(trajectory_file_CO_OH,format='extxyz',index=-1)

# Visualizziamo per essere sicuri
view(atoms_catalysis)

In che tipo di dato è immagazzinato il frame che abbiamo appena letto?

In [ ]:
# Usiamo type() per ottenere il tipo di dato (o tipo di oggetto)
type(atoms_catalysis)

In [ ]:
# La funzione help() mostra gli attributi e i metodi di Atoms
help(Atoms)

In [ ]:
print("Informazioni e metadati sul sistema (tutti gli atomi a un dato frame):\n",atoms_catalysis)
print("\nInformazioni sul primo atomo:\n",atoms_catalysis[0])

Come selezioniamo gli atomi per ciascun tipo di elemento? Si puo' usare `np.where()`:

In [ ]:
# In generale np.where() lavora su array 2D o 3D, dunque restituisce una tuple
# Noi vogliamo solo il primo elemento della tuple -> [0]
ind_H = np.where(atoms_catalysis.symbols=='H')[0]
ind_C = np.where(atoms_catalysis.symbols=='C')[0]
ind_O = np.where(atoms_catalysis.symbols=='O')[0]
ind_Cu = np.where(atoms_catalysis.symbols=='Cu')[0]

In alternativa, si può usare **list comprehension**:

In [ ]:
ind_H = [a.index for a in atoms_catalysis if a.symbol == 'H']
ind_C = [a.index for a in atoms_catalysis if a.symbol == 'C']
ind_O = [a.index for a in atoms_catalysis if a.symbol == 'O']
ind_Cu = [a.index for a in atoms_catalysis if a.symbol == 'Cu']

In [ ]:
print(ind_O)

In [ ]:
for i in ind_O :
    print(atoms_catalysis[i])

Troviamo dunque due atomi di ossigeno, rispettivamente agli indici 36 e 39.

## Simulazione NVE con MACE e Velocity Verlet

Per prima cosa testeremo la stabilità numerica di MACE eseguendo una breve simulazione in ensemble NVE (numero di atomi, volume ed energia totale costanti).

A tal fine, utilizzeremo un modello "relativamente piccolo" (~8 milioni di pesi) ma molto potente: `mace-omat-0-small.model`. Qui trovate la lista dei _foundation models_ di MACE: https://github.com/ACEsuit/mace-foundations

La versione di MACE OMAT è considerata un _foundation model_, ovvero un modello capace di simulare (quasi) qualsiasi combinazione di specie atomiche (89 elementi). È stato allenato sul dataset [**O**pen **Mat**erials](https://huggingface.co/datasets/facebook/OMAT24) di Meta (FairChem).

<img src="img/OMAT-elements.png" width="550">

Fonte: https://arxiv.org/abs/2410.12771v1

Per prima cosa scarichiamo il modello dalla repository GitHub di MACE:

In [ ]:
!wget https://github.com/ACEsuit/mace-foundations/releases/download/mace_omat_0/mace-omat-0-small.model

Qui entra in gioco la natura di "orchestratore" di ASE: definiamo un calcolatore di interazioni interatomiche usando la libreria Python di MACE: 

In [ ]:
from mace.calculators import MACECalculator

# Il file che abbiamo appena scaricato
model_file = "mace-omat-0-small.model"

# Ignorate questo parametro per il momento
model_head = None

calc_mace = MACECalculator(model_paths=model_file,
                           head=model_head,
                           default_dtype="float32")

Il calcolatore MACE può essere "assegnato" a un tipo `Atoms` di ASE come attributo (`Atoms.calc`): ogniqualvolta ASE richiederà una forza o un energia, sarà `MACECalculator` a calcolarla e restituirla ad `Atoms`.

In [ ]:
# Leggiamo nuovamente l'ultimo frame della traiettoria 
# (non necessario, serve solo come reset)
atoms_catalysis = read(trajectory_file_CO_OH,format='extxyz',index=-1)

# Assegnamo il calcolatore MACE al sistema di atomi
atoms_catalysis.calc = calc_mace

# Calcoliamo e stampiamo l'energia potenziale del sistema
print("Energia totale =",atoms_catalysis.get_potential_energy(),"eV")

### Step 1 - Minimizzazione

Cerchiamo prima di trovare un minimo locale dell'energia potenziale, usando l'ottimizzatore di Broyden-Fletcher-Goldfarb-Shanno (BFGS). E' equivalente ad una simulazione di _energy minimization_ (simile a `integrator=sd` in GROMACS).

Le energie (e le forze) verranno calcolate da `MACECalculator`.

In [ ]:
from ase.optimize import BFGS

optimizer = BFGS(atoms_catalysis)
optimizer.run(fmax=1e-2,steps=200)

### Step 2 - Dinamica

Prima di partire con la simulazione molecolare, inizializziamo le velocità degli atomi. Assicuriamoci inoltre che la velocità totale del sistema sia nulla (i.e. non ci sia _drift_ del centro di massa):

In [ ]:
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution
from ase.md.velocitydistribution import Stationary

# Prendiamo le velocita' da una distribuzione di Boltzmann
# come se la temperatura iniziale del sistema sia T(0)=300K
MaxwellBoltzmannDistribution(atoms_catalysis, temperature_K=300)

# Sottraiamo la velocita' totale
Stationary(atoms_catalysis)

La **dinamica** molecolare richiede un metodo di integrazione numerica, oltre ad un calcolatore di interazioni interatomiche. Ad esempio, ASE implementa nativamente il metodo di [Velocity Verlet](https://en.wikipedia.org/wiki/Verlet_integration#Velocity_Verlet):

$$ \vec{x}(t+\Delta t) = \vec{x}(t) + \vec{v}(t)\Delta t + \frac{\vec{a}(t)}{2}\Delta t^2 \implies \mbox{MACE calcola : }\vec{a}(t+\Delta t) \implies \vec{v}(t+\Delta t) = \vec{v}(t) + \frac{\vec{a}(t)+\vec{a}(t+\Delta t)}{2}\Delta t$$

dove semplicemente $\vec{a}=\vec{F}/m$. Notare ancora una volta la separazione dei lavori: MACE si occuperà di calcolare le forze, o equivalentemente le accelerazioni $\vec{a}$ (pesante e difficile), ASE si occuperà di calcolare l'update di velocità $\vec{v}$ e posizioni $\vec{x}$ (leggero e relativamente semplice).

In [ ]:
from ase.md.verlet import VelocityVerlet
from ase import units

# Tempo totale della simulazione
T_FIN = 500 # fs

# Time step
dt = 1.0 # fs

### TODO: Per la relazione vi verrà chiesto di provare timestep differenti ###
# dt = ...

# Passiamo a VelocityVerlet il sistema atomico e il time step
integrator_vv = VelocityVerlet(atoms_catalysis, 
                               dt*units.fs)

In [ ]:
# Possiamo dare un'occhiata a come è fatto VelocityVerlet
help(VelocityVerlet)

Ora una parte un po' "noiosa": definire come viene gestito l'output della simulazione

In [ ]:
# Costante di conversione da femtosecondi e picosecondi
FS2PS = 1e-3

# File dove verrà scritta la traiettoria
dump_file = "md_test.xyz"

# Vettori dove salvare tempi ed energie
time_ps = []
pot_energy = []
kin_energy = []
tot_energy = []

def write_frame():
    integrator_vv.atoms.write(dump_file, append=True)
    time_ps.append(FS2PS*integrator_vv.get_time()/units.fs)
    pot_energy.append(integrator_vv.atoms.get_potential_energy())
    kin_energy.append(integrator_vv.atoms.get_kinetic_energy())
    tot_energy.append(integrator_vv.atoms.get_total_energy())

La funzione `write_frame()` può essere assegnata a `VelocityVerlet` usando il metodo `attach()`; verrà chiamata ogni dato intervallo di tempo.

Ci siamo quasi: definitamo gli ultimi parametri di output:

In [ ]:
# Costante: da femtosecondi a nanosecondi
FS2NS = 1e-6

# Costante: da giorni a minuti
DAY2MIN = 1440

# Costante: numero di frames in output
NTRAJ = 100

# Numero di step della simulazione
nsteps = int(T_FIN/dt)

# Frequenza di output ("dump")
fdump = nsteps//NTRAJ

In [ ]:
# Rimuoviamo la traiettoria, se esiste già (per non "appendere" ad una traiettoria vecchia)
!rm {dump_file}

Siamo finalmente pronti ad eseguire la nostra prima simulazione molecolare con neural-network potentials (!).

In ordine:
1. "Attacchiamo" i loggers come _callbacks_ all'integratore numerico (per gestire output su file e su stdout);
2. Definiamo delle variabili ausiliari per cronometrare la simulazione;
3. Eseguiamo `VelocityVerlet.run()` per far partire la simulazione.

In [ ]:
from ase.md.logger import MDLogger

# Logger: serve per stampare a schermo (stdout) la progressione della simulazione
stdout_logger = MDLogger(integrator_vv,
                         atoms_catalysis,
                         sys.stdout,
                         header=True,
                         stress=False,
                         peratom=False)

# "Attacchiamo" i loggers all'integratore numerico
integrator_vv.attach(stdout_logger, interval=fdump)
integrator_vv.attach(write_frame, interval=fdump)

# Timing: tempo di inizio
t0 = time.time()

# Questa riga esegue effettivamente la simulazione
# i.e. l'integratore numerico "propaga" il sistema per
# nsteps, stampando in output ogni fdump
integrator_vv.run(nsteps)

# Tempo di fine
t1 = time.time()
cpu_run_min = (t1-t0)/60

print("\n#################################")
print("MD finished in {0:.2f} minutes!".format(cpu_run_min))
print("Estimate: {0:.4f} ns/day".format(FS2NS*DAY2MIN*(nsteps*dt)/cpu_run_min))
print("#################################")

&#10067; | Ricordate qual'era l'ordine di grandezza della velocità di calcolo di GROMACS (ns/day), quando avete simulato CNT e zeolite?

Per convenienza di visualizzazione, sottraiamo alle energie il loro valore iniziale. Siamo interessati all'andamento temporale, non al valore assoluto:

In [ ]:
# Convertiamo a vettore NumPy e sottraiamo il primo valore
pot_energy = np.array(pot_energy)
kin_energy = np.array(kin_energy)
tot_energy = np.array(tot_energy)
pot_energy -= pot_energy[0]
kin_energy -= kin_energy[0]
tot_energy -= tot_energy[0]

In [ ]:
%matplotlib widget

fig1, ax1 = plt.subplots()
plt.plot(time_ps,pot_energy,'b--',label=r"$E_{pot}-E_{pot}(0)$")
plt.plot(time_ps,kin_energy,'r:',label=r"$E_{kin}-E_{kin}(0)$")
plt.plot(time_ps,tot_energy,'k-',label=r"$E_{tot}-E_{tot}(0)$")
plt.xlim([0,T_FIN*FS2PS])
plt.xlabel("t [ps]")
plt.ylabel("Energy [eV]")
plt.legend()
plt.show()

Come era lecito aspettarsi, l'energia totale del sistema rimane costante, mentre energia potenziale si trasforma in cinetica (e vice-versa). L'ensemble NVE rimane un **sistema Hamiltoniano**, pur avendolo simulato con un NNP:

$$ \frac{d\vec{p}}{dt} = -\frac{\partial H}{\partial \vec{x}} \; , \quad \frac{d\vec{x}}{dt} = \frac{\partial H}{\partial \vec{p}} $$

dove $\vec{p}=m\vec{v}$ e $H(\vec{x},\vec{p})=E_{pot}(\vec{x})+E_{kin}(\vec{p})=E_{tot}$.

Controllare che l'energia totale **in ensemble NVE** rimanga costante (al netto di "piccole" oscillazioni numeriche) è un modo per valutare la stabilità numerica della simulazione (i.e. se il time step `dt` è sufficientemente piccolo).

&#9888; **NB**: Notate come l'energia cinetica (i.e. temperatura) devii molto dal suo valore iniziale. Abbiamo simulation in ensemble NVE, quindi il sistema non e' termalizzato (i.e. non c'e' termostato). In sostanza si comporta come un sistema isolato.

Apriamo e visualizziamo la traiettoria:

In [ ]:
trajectory_vv = read(dump_file,format='extxyz',index=':')
view(trajectory_vv)

## Campionamento con NEB

**Buona notizia**: se siete arrivati fino a questo punto, significa che siete riusciti ad eseguire una simulazione usando _Neural-Network Potentials_, wow!

**Cattiva notizia**: la simulazione che abbiamo appena eseguito è servita a valutare la stabilità, nulla più. Anche lasciando girare la simulazione per ore, non vedremmo accadere nulla di interessante: il composto rimarrebbe nello stato COOH$^*$, senza tornare CO$^*$+OH$^*$ (perchè?).

Per campionare (_sample_) la reazione inversa, utilizzeremo il metodo della "catena elastica perturbata", [**N**udged **E**lastic **B**and (**NEB**)](https://doi.org/10.1063/1.1323224).

<img src="https://umet.univ-lille.fr/Projets/RheoMan/uploads/images/More/Fig3.jpg" width="450"/>

Fonte immagine: https://umet.univ-lille.fr/Projets/RheoMan/en/to-learn-more-about/nudged-elastic-band.php.html

In sostanza, vogliamo trovare il percorso migliore tra stato iniziale e finale (i.e. quello che minimizza l'energia massima).

**Ipotesi**: conosciamo lo stato iniziale e quello finale (i.e. abbiamo una configurazione per entrambi gli stati).

NEB funziona un questa maniera:
1. Definisce `n_images` stati intermedi tra $\text{COOH}^*$ e $\text{CO}^*+\text{OH}^*$;
2. Interpola le posizioni per ottenere una prima "guess" di percorso;
3. Per ogni stato intermedio (immagine), ottimizza/simula in modo da ottenere un minimo energetico locale, **soggetto ad un vincolo elastico (molla)** con l'immagine precedente e quella successiva.

<img src="img/NEB.png" width="900">

### Interpolazione

Leggiamo lo stato iniziale (corrispondente a $\mbox{COOH}^*$) e lo stato finale (corrispondente a $\mbox{CO}^*+\mbox{OH}^*$).

In [ ]:
initial = read(trajectory_file_CO_OH,format='extxyz',index=-1)
final = read(trajectory_file_CO_OH,format='extxyz',index=0)

In [ ]:
# Ri-definiamo MACE
# Non strettamente necessario, evitiamo di copiare su vecchi oggetti)
model_file = "mace-omat-0-small.model"
model_head = None
calc_mace = MACECalculator(model_paths=model_file,head=model_head,default_dtype="float32")

# Notare l'uso di copy(): vogliamo usare un calcolatore DISTINTO per ogni stato, 
# in modo da non sovrascrivere le energie
initial.calc = copy.copy(calc_mace)
final.calc = copy.copy(calc_mace)

# Minimizziamo le due configurazioni iniziali
optimizer_init = BFGS(initial)
optimizer_init.run(fmax=1e-2,steps=100)
optimizer_fin = BFGS(final)
optimizer_fin.run(fmax=1e-2,steps=100)

Importiamo l'oggetto `NEB` e diamo un'occhiata alla documentazione:

In [ ]:
from ase.mep.neb import NEB

help(NEB)

In [ ]:
# Definitamo la lista di stati intermedi
n_images = 7
images = [initial]
images += [initial.copy() for i in range(n_images-2)]
images += [final]

# Definiamo l'oggetto NEB, 
# specificando la costante elastica e il metodo di interpolazione
neb = NEB(images, 
          k=0.5, 
          method='string')

# Interpoliamo per creare la prima "guess" del percorso
# IDPP = Image Dependent Pair Potential, migliora l'interpolazione iniziale
neb.interpolate(method='idpp')

### Ottimizzazione

Associamo un calcolatore MACE ad ogni immagine. Verra' chiamato dall'ottimizzatore per calcolare energie e forze:

In [ ]:
# Notare l'utilizzo di copy()
# L'ottimizzatore vuole un'istanza diverse dall'oggetto per ogni immagine!
for image in images[1:n_images-1]:
    image.calc = copy.copy(calc_mace)

E ora la parte _lenta_: per ogni iterazione, l'ottimizzatore opera uno step di minimizzazione dell'energia di tutta la catena (i.e. tutti gli stati, soggetti ai vicoli elastici tra di loro).

In [ ]:
from ase.optimize import BFGS
from ase.io import Trajectory

optimizer = BFGS(neb, trajectory='B2A.traj')
for i in range(n_images):
    optimizer.attach(Trajectory('B2A-%d.traj' % i, 'w', images[i]))

# Questi numeri sono molto "ottimistici": in realta' la tolleranza dovrebbe essere
# molto piu' bassa (e.g. 1e-3) e il numero di steps molto piu' alto (e.g. >1000), 
# ma il laboratorio dura 1.5 ore, non 1.5 giorni
optimizer.run(fmax=0.1,steps=50)

Piccola digressione tecnica: la traiettoria in output è aggregata in questo modo ("frame major"):

    {(image_0,frame_0), (image_1,frame_0), ..., (image_n,frame_0), (image_0,frame_1) ..., (image_n,frame_m)}
    
Per poterla visualizzare più facilmente, meglio trasporre in quest'altro modo ("image major"):

    {(image_0,frame_0), (image_0,frame_1), ..., (image_0,frame_m), (image_1,frame_0) ..., (image_n,frame_m)}

In [ ]:
from ase.io import write, Trajectory
traj = read("B2A.traj", index=":")
# Piu' semplice scrivere in formato testuale .xvg
# rispetto al formato binario .traj
with open("B2A_ordered.xyz", "w") as f:
    for i in range(n_images):
        for k in range(i, len(traj), n_images):
            write(f, traj[k], format="xyz", append=True)

Visualizziamo gli stati intermedi, uno dopo l'altro:

In [ ]:
traj_all_images = read("B2A_ordered.xyz", index=":")

view(traj_all_images)

### Coordinata di reazione

Vogliamo ora ottenere il profilo di energia potenziale sulla catena di stati.

Ogni stato è descritto da un insieme di coordinate atomiche; l'energia potentiale dello stato $i$ e' una funzione di queste coordinate:

$$ E_{pot}^i = E_{pot}^i(\vec{x}_i) $$

È scomodo associare il potenziale a **tutte** le coordinate del sistema (pesante e non può essere visualizzato facilmente). Meglio associare il potenziale ad una descrizione ridotta del sistema, anche chimata **coordinata di reazione** (_reaction coordinate_), o anche **variabile collettiva** (_collective variable_).

$$ E_{pot}^i = E_{pot}^i(x_i^{cv}) $$

In sostanza dobbiamo operare una riduzione di dimensionalità (wink wink). Possiamo:
1. Usare l'intuizione chimica;
2. Usare un metodo di machine learning (e.g. PCA).

<img src="img/cv-COOH.png" width="400">

Usando l'intuizione chimica, possiamo definire come coordinata di reazione la distanza tra l'atomo di carbonio in **C**O$^*$e e l'atomo di ossigeno di **O**H$^*$:

$$d_{CO}=||\boldsymbol{x}_{C}-\boldsymbol{x}_{O}||$$

In [ ]:
# Salviamo l'ultimo frame della traiettoria di ogni immagine
min_image = []
for i in range(n_images) :
    traj_temp = read(f"B2A-{i}.traj".format(i), index=":")
    min_image.append(traj_temp[-1])

Visualizziamo uno degli stati per trovare le coordinate degli atomi che ci interessano:

In [ ]:
view(min_image[-1])

In [ ]:
index_C_cv = 37
index_O_cv = 36

### TODO: Cambiando traiettoria, potrebbero cambiare anche gli indici ##
# index_C_cv = 
# index_O_cv = 

Creiamo una lista per la _collective variable_ (CV) e l'energia potenziale, e salviamo un valore per ogni immagine:

In [ ]:
from numpy.linalg import norm

dist_cv_neb = []
pot_energy_cv_neb = []
for i in range(n_images) :
    x_C = min_image[i][index_C_cv].position
    x_O = min_image[i][index_O_cv].position
    d_CO = norm(x_C-x_O,2)
    dist_cv_neb.append(d_CO)
    min_image[i].calc = copy.copy(calc_mace)
    pot_energy_cv_neb.append(min_image[i].get_potential_energy())
    
# Cast ad array NumPy (per convenianza)
dist_cv_neb = np.array(dist_cv_neb)
pot_energy_cv_neb = np.array(pot_energy_cv_neb)

# Sottraggo il minimo del profilo del potenziale (il minimo diventa 0)
pot_energy_cv_neb -= np.min(pot_energy_cv_neb)

Per avere un riferimeno, calcoliamo anche il profilo di potenziale della traiettoria originale (articolo). L'energia potenziale della simulazione originale è salvata nei metadati della traiettoria.

**NB** - La direzione temporale di lettura della traiettoria non ha importanza: stiamo valutando potenziale vs. CV, la dimensione temporale scompare.

In [ ]:
trajectory_catalysis = read(trajectory_file_CO_OH,format='extxyz',index=':')
dist_cv_ref = []
pot_energy_cv_ref = []
for atoms in trajectory_catalysis:
    x_C = atoms[index_C_cv].position
    x_O = atoms[index_O_cv].position
    d_CO = norm(x_C-x_O,2)
    dist_cv_ref.append(d_CO)
    atoms.calc = calc_mace
    pot_energy_cv_ref.append(atoms.get_potential_energy())
dist_cv_ref = np.array(dist_cv_ref)
pot_energy_cv_ref = np.array(pot_energy_cv_ref)
pot_energy_cv_ref -= np.min(pot_energy_cv_ref)

In [ ]:
%matplotlib widget

fig2, ax2 = plt.subplots()
plt.plot(dist_cv_ref,pot_energy_cv_ref,'ko', label='Reference (article)')
plt.plot(dist_cv_neb,pot_energy_cv_neb,'rx:', label='NEB')
plt.legend()
plt.xlabel(r"$d_{CO}$ [Å]")
plt.ylabel(r"$E_{pot}$ [eV]")
plt.show()

## MACE vs. DFT

Nell'ultima parte dell'esercizio, esaminiamo la seconda reazione di elettro-ossidazione, i.e. la de-protonazione di $\mbox{COOH}^*$ (che diventa $\mbox{CO}_2$) e la protonazione di un ulteriore gruppo $\mbox{OH}^*$ (che diventa $\mbox{H}_2\mbox{O}$):

$$ \mbox{COOH}^* + \mbox{OH}^* \rightleftharpoons \mbox{CO}_2 + \mbox{H}_2\mbox{O} $$

Invece di simulare, ci limiteremo ad estrarre l'energia potenziale dalla traiettoria già disponibile, per confrontarla con una simulazione **Density Functional Theory (DFT)** di riferimento.

In [ ]:
trajectory_file_CO2_H2O = "COoxCu_111_COO-H.extxyz"

### TODO: Per la relazione dovrete caricare una traiettoria differente ###
# trajectory_file_CO2_H2O = ...

trajectory_catalysis = read(trajectory_file_CO2_H2O,format='extxyz',index=':')

In [ ]:
view(trajectory_catalysis)

### Modelli _multi-head_

Cambiamo modello e usiamo un nuovo _foundation model_, chiamato **MACE-MH** (MH=**M**ulti-**H**ead).

I dettagli sul NNP sono consultabili qui: https://huggingface.co/mace-foundations/mace-mh-1

<img src="img/multihead.png" width="650">

Fonte: https://arxiv.org/abs/2510.25380v1

I modelli _multi-head_ sono la composizione di un modello "base" + una (o più) "head". Questa architettura è vantaggiosa per operare _fine-tuning_, i.e. aggiornare i pesi della rete neurale alla luce di un dataset specifico (ri-calibrando principalmente, o soltando, i pesi della head). 

È quindi possibile allenare heads diverse su training set diversi, per rendere il modello più accurato in diversi ambiti:
- Molecole organiche;
- MOF;
- Superfici, interfacce, catalisi;
- Liquidi (e.g. acqua);
- Biochimica (proteine, ligandi);
- ...

In [ ]:
# Scarichiamo il modello MH-1
!wget https://github.com/ACEsuit/mace-foundations/releases/download/mace_mh_1/mace-mh-1.model

Usiamo la head "default" `omat_pbe`, che dovrebbe garantire buona accuratezza per materiali e reazioni generiche.

In [ ]:
model_file = "mace-mh-1.model"

model_head = "omat_pbe"

### TODO: Per la relazione, vi verrà chiesto di cambiare head ###
# model_head = ...

calc_mace = MACECalculator(model_paths=model_file,
                           head=model_head,
                           default_dtype="float32")

Per curiosità, ispezioniamo il numero di parametri del modello:

In [ ]:
model = calc_mace.models[0]

base_params = []
head_params = []
for name, p in model.named_parameters():
    if "readout" in name:
        head_params.append(p)
    else:
        base_params.append(p)
print("#parameters (base):", sum(p.numel() for p in base_params))
print("#parameters (head):", sum(p.numel() for p in head_params))

Ri-calcoliamo le energie sulla traiettoria. Questo passaggio (i.e. calcolare le energie su frame noti, senza lo step di update temporale) è molto comune nei metodi di _active learning_, dove si vuole confrontare l'energia degli stati di una traiettoria vs. DFT oppure tra più NNPs (_query-by-committee_).

Le energie della simulazione DFT originale sono scritte nei metadati della traiettoria (`DFT_energy`). Ignorate invece i metadati associati a `MACE_energy`, fanno riferimento al modello MACE del paper da cui è stata tratta la traiettoria.

In [ ]:
n = 0
frame = []
potential_mace = []
potential_dft = []

for atoms in trajectory_catalysis:
    
    atoms.calc = calc_mace
    print("Frame",n,", info:",atoms.info)
    potential_dft.append(atoms.info['DFT_energy'])
    n += 1
    frame.append(n)
    potential_mace.append(atoms.get_potential_energy())

In [ ]:
# Come al solito, "castiamo" a vettore NumPy e scaliamo
potential_mace = np.array(potential_mace)
potential_dft = np.array(potential_dft)
potential_mace -= potential_mace[0]
potential_dft -= potential_dft[0]

In [ ]:
%matplotlib widget

fig3, ax3 = plt.subplots()
plt.plot(frame, potential_dft, 'k:.', label="DFT (article)")
plt.plot(frame, potential_mace, 'r:x', label="MH ("+model_head+")")
plt.xlabel('frame []')
plt.ylabel('potential energy [eV]')
plt.legend()
plt.show()

In [ ]:
### TODO: Vi verrà chiesto di definire una nuova variabile di reazione ###

### Discussione

MACE riesce a ricostruire il profilo di energia potenziale in maniera eccellente. C'è però un piccolo _caveat_: la superficie $\text{Cu (111)}$ è nel dataset di training di `omat_pbe`, quindi una buona ricostruzione delle energie di reazione è comprensibile.

Cosa succede quando cambiamo il piano di taglio (i.e. disposizione degli atomi sulla superifcie) o la _head_ del modello?